In [23]:
import pandas as pd

df = pd.read_csv("./../data/corrected_phishing_dataset.csv")

print("Liczba rekordów:", df.shape[0])
print("Liczba kolumn:", df.shape[1])

print("\nInformacje o kolumnach:")
df.info()
TARGET = "type"

Liczba rekordów: 651191
Liczba kolumn: 2

Informacje o kolumnach:
<class 'pandas.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   url     651191 non-null  str  
 1   type    651119 non-null  str  
dtypes: str(2)
memory usage: 9.9 MB


In [24]:
print("\nLiczba brakujących wartości w każdej kolumnie:")
missing = df.isna().sum()
missing = missing[missing>0]
print(missing.to_string())


Liczba brakujących wartości w każdej kolumnie:
type    72


In [25]:
print("Przed:", len(df))

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("Po:", len(df))
print("Brakujące etykiety:", df[TARGET].isna().sum())

Przed: 651191
Po: 651119
Brakujące etykiety: 0


In [26]:
print("Duplikaty URL:", df.duplicated(subset=["url"]).sum())
conflicting_labels = (
    df.groupby("url")[TARGET]
    .nunique()
)

print(
    "URL z różnymi etykietami:",
    (conflicting_labels > 1).sum()
)

Duplikaty URL: 10072
URL z różnymi etykietami: 0


In [27]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Usunięte duplikaty:", before - len(df))
print("Pozostałe rekordy:", len(df))

Usunięte duplikaty: 10072
Pozostałe rekordy: 641047


In [28]:
df[TARGET].value_counts()

type
benign        428182
defacement     95308
phishing       93912
malware        23645
Name: count, dtype: int64

In [29]:
import re

urls = df["url"]

http_count = urls.str.match(
    r"^http://", flags=re.IGNORECASE, na=False
)

https_count = urls.str.match(
    r"^https://", flags=re.IGNORECASE, na=False
)

no_protocol_count = len(df) - http_count.sum() - https_count.sum()

print(f"HTTP: {http_count.sum()}")
print(f"HTTPS: {https_count.sum()}")
print(f"Bez HTTP/HTTPS: {no_protocol_count}")

HTTP: 162317
HTTPS: 15637
Bez HTTP/HTTPS: 463093


In [30]:
import re
from urllib.parse import urlsplit
from IPython.display import display

SCHEME_PATTERN = re.compile(r"^[A-Za-z][A-Za-z0-9+.-]*://")
CONTROL_PATTERN = re.compile(r"[\x00-\x1F\x7F-\x9F]")

VALID_LABELS = {
    "benign",
    "phishing",
    "malware",
    "spam",
    "defacement"
}


def is_valid_url(url):
    if not isinstance(url, str) or not url.strip():
        return False

    if CONTROL_PATTERN.search(url):
        return False

    try:
        if SCHEME_PATTERN.match(url) or url.startswith("//"):
            parsed = urlsplit(url)
        else:
            parsed = urlsplit("//" + url)
        return bool(parsed.hostname)

    except ValueError:
        return False


# 1. Walidacja adresów URL
valid_url_mask = df["url"].apply(is_valid_url)

valid_mask = valid_url_mask

# 4. Wyodrębnienie odrzuconych rekordów
df_invalid = df.loc[~valid_mask].copy()

print("Liczba wszystkich rekordów:", len(df))
print("Nieprawidłowe URL-e:", (~valid_url_mask).sum())
print("Łącznie odrzuconych rekordów:", len(df_invalid))

display(df_invalid[["url", "type"]])

# 5. Usunięcie nieprawidłowych rekordów
df = df.loc[valid_mask].copy().reset_index(drop=True)

print("Rekordy po filtrowaniu:", len(df))

Liczba wszystkich rekordów: 641047
Nieprawidłowe URL-e: 34
Nieprawidłowe etykiety: 0
Łącznie odrzuconych rekordów: 34


,url,type
61370,http://www.scsrxx.com/js/?https://secure.runes...,phishing
520493,http://gkjx168.com/images?us.battle.net/login/...,phishing
523151,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523240,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523243,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523244,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523245,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523255,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523256,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523259,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing


Rekordy po filtrowaniu: 641013


In [33]:
#Czyszczenie rekordów zaczynających się znakiem apostrofu, nie były poprawnie parsowane przez spliturl
# Identyfikacja rekordów do poprawienia
mask = df["url"].str.startswith("'") & \
       df["url"].str.endswith("'")

print("Liczba URL-i wymagających poprawienia:", mask.sum())
display(df.loc[mask, ["url", "type"]].head(20))
df["url"] = df["url"].str.strip("'")

Liczba URL-i wymagających poprawienia: 3899


,url,type
545184,'motors.shop.ebay.com-cars-trucks-724527.jnq3....,phishing
545198,'www.gestion-des-impayes.com/visuel.php?param=...,phishing
545275,'nicolecustodio.com.br/paypal\%20us/webscr.htm...,phishing
545361,'beforenanny911.com/auto/my-themes/file/proper...,phishing
545368,'www.edyshsdf32.hut4.ru/Redirecionamento.html?...,phishing
545408,'www.google.ad/url?sa=f&rct=j&url=https:/www.p...,phishing
545492,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545498,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545540,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545541,'justoneofthosethings.co.uk/bahi\%20paypal/',phishing


In [34]:
df.to_csv('./../data/deduplicated_corrected_phishing_dataset.csv',index=False, encoding="utf-8")